In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
from openai import OpenAI

load_dotenv()

GROQ_MODEL = "llama-3.1-8b-instant"
groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [23]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=messages,
)

response.choices[0].message.content

"You're referring to a course, but I'm a large language model, I don't have specific information about your course. Can you please provide more details about the course, such as its name or where you found it? That way, I can try to help you better or provide general information about the course registration process."

In [4]:
messages = [
    {"role": "user", "content": "How do I run Ollama locally?"}
]

response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=messages,
)

response.choices[0].message.content

"Ollama is a highly customizable AI assistant developed by Jason Weston and Ellie Pavlick (previously known as LLaMA by Meta). To run Ollama locally, you'll need to follow these steps:\n\n**Prerequisites:**\n\n1. **Python 3.9 or higher**: Ollama requires Python 3.9 or higher to run.\n2. **pip**: Make sure you have the latest version of pip installed.\n3. **TensorFlow 2.9 or higher**: Ollama uses TensorFlow 2.9 or higher for deep learning operations.\n4. **Transformers 4.20 or higher**: Ollama utilizes the Transformers library for model loading and inference.\n5. **LLaMA model weights**: You'll need to download the LLaMA model weights, which are around 13 GB in size.\n\n**Step-by-Step Instructions:**\n\n1. **Install required dependencies**: Run the following command in your terminal or command prompt to install the necessary dependencies:\n```bash\npip install transformers tensorflow\n```\n2. **Download the LLaMA model weights**: Clone the Ollama repository from GitHub:\n```bash\ngit cl

In [5]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=instructions,
    model=GROQ_MODEL,
)

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, you need to first install it. The steps to install Ollama are:

* For **macOS**, download the `.pkg` and install it.
* For **Windows**, download the `.msi` and install it.
* For **Linux**, run the following command in the terminal: ```bash
curl -fsSL https://ollama.com/install.sh | sh
```

Once installed, open a terminal and type:

```
ollama run llama3
```

This command will:

* Download the LLaMA 3 model (~4GB).
* Start the model locally.
* Open a chat-like interface where you can type questions.

To test the Ollama local server, run the following command:

```
curl http://localhost:11434
```

This should give you a response similar to:

```
{"models": [...]}
```


In [8]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

Based on the context provided, it seems like Olama is a module similar to 'RAG'. However, since there is no information about running 'Olama' locally in the context provided, I will answer the question based on the similar context from the provided FAQ.

Given the context where we can run RAG locally by setting up Python, uv, Jupyter, Docker, and other tools needed for the module, a possible answer might be similar for 'Olama'. However, this information is not available in the context, so I will use the given information to advise on how to proceed.

To run Olama locally, you may need to follow a similar setup as for RAG or other modules, but unfortunately, specific instructions are not available. 

Please note that I have to be cautious in my answer since I do not have explicit information about Olama, and any information about it was not provided within this context. To determine the correct setup, I recommend checking with the course instructors for specific instructions on setting 

Define the function to use as a tool

In [24]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [25]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

In [26]:
text = messages[0]["content"]

In [41]:
text

'I just discovered the course. Can I join it?'

In [27]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. You have access to a search function to look up information in a course FAQ database. Use it when appropriate to answer questions."},
        {"role": "user", "content": text}
    ],
    tools=[search_tool],
    tool_choice="auto",
)

In [28]:
parts = response.choices[0].message.tool_calls or []

In [29]:
len(parts)

1

In [30]:
parts

[ChatCompletionMessageFunctionToolCall(id='qms4vkayp', function=Function(arguments='{"query":"course joining policy"}', name='search'), type='function')]

In [31]:
import json

call = parts[0] if parts else None

if not call:
    args = {"query": text}
else:
    args = json.loads(call.function.arguments)

In [32]:
args

{'query': 'course joining policy'}

In [33]:
call.function.name if call else "search"

'search'

In [34]:
results = search(**args)

In [35]:
results

[{'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offered next?',
  'answer': 'Summer 2027.'},
 {'id': '04919992b3',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Sub

In [36]:
result_json = {
    "results": results   # <-- wrap list
}

Function Calling with Groq (OpenAI-compatible API)

1. Define your tool schema

In [37]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database...",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "..."}
            },
            "required": ["query"]
        }
    }
}

2. Make initial request with tool

In [45]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Use the search function to look up relevant information in the course FAQ database before answering questions."},
        {"role": "user", "content": text}
    ],
    tools=[search_tool],
    tool_choice="auto",
)

3. Extract and execute the function call

In [46]:
import json

parts = response.choices[0].message.tool_calls or []
call = parts[0] if parts else None

if not call:
    print(response.choices[0].message.content)
    args = {"query": text}
else:
    args = json.loads(call.function.arguments)

result = search(**args)

4. Send results back (multi-turn)

In [47]:
import json

if call:
    assistant_message = {
        "role": "assistant",
        "content": response.choices[0].message.content or "",
        "tool_calls": [
            {
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.function.name,
                    "arguments": call.function.arguments,
                },
            }
        ],
    }

    tool_message = {
        "role": "tool",
        "tool_call_id": call.id,
        "content": json.dumps({"results": result}),
    }

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[
            {"role": "user", "content": text},
            assistant_message,
            tool_message,
        ],
    )

print(response.choices[0].message.content)

Yes, you can join the course. However, if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. You can start the course whenever you want, and the videos and GitHub materials are available. A typical workflow is to watch the lesson videos, work through the lesson notebooks/code, read the homework instructions on GitHub, and submit answers through the course platform before the deadline.


In [48]:
response.usage

CompletionUsage(completion_tokens=86, prompt_tokens=699, total_tokens=785, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.018286029, prompt_time=0.040695334, completion_time=0.108311734, total_time=0.149007068)

In [49]:
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

In [50]:
def estimate_cost(input_tokens, output_tokens,
                  input_price=0.15,   # per 1M tokens (example Flash rate)
                  output_price=1.25): # per 1M tokens

    cost = (input_tokens / 1_000_000 * input_price) + \
           (output_tokens / 1_000_000 * output_price)

    return cost

In [51]:
response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": "Explain what RAG is in simple terms"}],
)

input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

cost = estimate_cost(input_tokens, output_tokens)

print("Input tokens:", input_tokens)
print("Output tokens:", output_tokens)
print("Estimated cost ($):", cost)

Input tokens: 44
Output tokens: 171
Estimated cost ($): 0.00022035000000000002


### The Agentic Loop

In [52]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [65]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

messages = [
    {'role': 'system', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [67]:
from openai import BadRequestError

try:
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        tools=[search_tool],
        tool_choice="auto",
    )
except BadRequestError as e:
    print("Tool calling failed; falling back to non-tool response.")
    print(e)
    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
    )

In [56]:
response

ChatCompletion(id='chatcmpl-56695003-202d-4d33-8dab-b1547ceb2ce2', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='I don\'t have any information on active courses. I can suggest searching the course catalog for available courses that might match your interests. <function=search>{"query": "course catalog"}</function>', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1782076261, model='llama-3.1-8b-instant', object='chat.completion', moderation=None, service_tier='on_demand', system_fingerprint='fp_4387d3edbb', usage=CompletionUsage(completion_tokens=40, prompt_tokens=222, total_tokens=262, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.089507469, prompt_time=0.013476469, completion_time=0.083830015, total_time=0.097306484), usage_breakdown=None, x_groq={'id': 'req_01kvp0bh39fjkra88n9e0x9es3', 'seed': 1927017753})

In [64]:
response.choices[0].message

ChatCompletionMessage(content='I don\'t have any information on active courses. I can suggest searching the course catalog for available courses that might match your interests. <function=search>{"query": "course catalog"}</function>', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [68]:
import json

assistant_msg = response.choices[0].message
tool_calls = assistant_msg.tool_calls or []

messages.append({
    "role": "assistant",
    "content": assistant_msg.content or "",
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments,
            },
        }
        for tc in tool_calls
    ],
})

if not tool_calls:
    # Fallback path: some Groq responses return pseudo function text instead of structured tool_calls.
    print("ASSISTANT (no structured tool_calls):")
    print(assistant_msg.content)
    print("\nFALLBACK (FAQ RAG):")
    print(assistant.rag(question))
else:
    for tc in tool_calls:
        print("function_call:", tc.function.name, tc.function.arguments)

        args = json.loads(tc.function.arguments or "{}")
        if tc.function.name == "search":
            tool_result = search(**args)
        else:
            tool_result = {"error": f"Unknown tool: {tc.function.name}"}

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        })

    follow_up = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
    )
    print("ASSISTANT:")
    print(follow_up.choices[0].message.content)

function_call: search {"query":"course join discovery"}
function_call: search {"query":"course registration timeline"}
ASSISTANT:
You've discovered the course and you're interested in joining it. You can join the course, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions. The course is self-paced, so you can start whenever you want, but you'll need to follow the weekly workflow, which includes watching lesson videos, working through lesson notebooks/code, reading homework instructions on GitHub, and submitting answers through the course platform before the deadline.

If you have any other questions or areas you'd like to explore, feel free to ask!


In [70]:
import json

for it in range(5):
    print(f"iteration #{it + 1}...")

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        tools=[search_tool],
        tool_choice="auto",
    )

    assistant_msg = response.choices[0].message
    tool_calls = assistant_msg.tool_calls or []

    # Keep the assistant turn in history for coherent follow-up generations.
    messages.append({
        "role": "assistant",
        "content": assistant_msg.content or "",
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in tool_calls
        ],
    })

    if not tool_calls:
        print("ASSISTANT:")
        print(assistant_msg.content)
        break

    for tc in tool_calls:
        print("function_call:", tc.function.name, tc.function.arguments)

        args = json.loads(tc.function.arguments or "{}")
        if tc.function.name == "search":
            tool_result = search(**args)
        else:
            tool_result = {"error": f"Unknown tool: {tc.function.name}"}

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(tool_result),
        })

iteration #1...
function_call: search {"query":"course join"}
iteration #2...
ASSISTANT:
You can join the course, but keep in mind that if you want to receive a certificate, you need to submit your project while the course is still accepting submissions. The course is offered next in Summer 2027. If you're new to the course, it's recommended to start with the LLM Zoomcamp docs, the general Zoomcamp logistics docs, and the LLM Zoomcamp GitHub repository.

Do you want to explore any other areas of the course, or would you like me to look up something specific for you?


In [71]:
messages

[{'role': 'system',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function.\nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'assistant',
  'content': '',
  'tool_calls': [{'id': '39pe915bp',
    'type': 'function',
    'function': {'name': 'search', 'arguments': '{"query":"course join"}'}}]},
 {'role': 'tool',
  'tool_call_id': '39pe915bp',
  'content': '[{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, 

In [87]:
def agent_loop(instructions, question, model=GROQ_MODEL) -> str:
    from openai import BadRequestError
    import json

    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ]

    last_answer = ""

    for it in range(1, 6):
        print(f"iteration #{it}...")

        try:
            response = groq_client.chat.completions.create(
                model=model,
                messages=messages,
                tools=[search_tool],
                tool_choice="auto",
            )
        except BadRequestError as e:
            print("Tool calling failed; using FAQ fallback.")
            print(e)
            last_answer = assistant.rag(question)
            print("ASSISTANT:")
            print(last_answer)
            break

        assistant_msg = response.choices[0].message
        tool_calls = assistant_msg.tool_calls or []

        messages.append({
            "role": "assistant",
            "content": assistant_msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in tool_calls
            ],
        })

        if not tool_calls:
            print("ASSISTANT:")
            last_answer = assistant_msg.content or ""
            print(last_answer)
            break

        for tc in tool_calls:
            print("function_call:", tc.function.name, tc.function.arguments)
            args = json.loads(tc.function.arguments or "{}")

            if tc.function.name == "search":
                tool_result = search(**args)
            else:
                tool_result = {"error": f"Unknown tool: {tc.function.name}"}

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(tool_result),
            })

    return last_answer

In [88]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [89]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"course join requirements"}
function_call: search {"query":"course registration process"}
function_call: search {"query":"what are the prerequisites for this course"}
iteration #2...
ASSISTANT:
You can still join the course, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions. The course is self-paced, and you can start whenever you want. However, if you want to receive a certificate, you need to finish the course with a "live" cohort.

Regarding the course materials, you can start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp). The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.

In [84]:
result

'Yes, you can join the course. However, if you want to receive a certificate, note the following conditions:\n\n1. The course\'s submission form must still be open when you submit your project.\n2. You need to finish the course with a "live" cohort to be eligible for a certificate.\n3. You can\'t get a certificate if you follow the course in a self-paced mode.'

In [81]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
function_call: search {"query":"bessie gould queen gambit"}
function_call: search {"query":"beth harmon queen gambit"}
iteration #2...
ASSISTANT:
It seems that the term "queen gambit" is not directly related to the course or its logistics. If you would like, I can try to provide more general information about the topic, or we could move on to another area that you would like to explore.


### ToyAIkit

In [91]:
!pip install toyaikit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 43.5 MB/s  0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [toyaikit]6/8 [anthropic]es]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [92]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [93]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [94]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [95]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [96]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [97]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [99]:
os.environ["OPENAI_API_KEY"] = os.getenv("GROQ_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = "https://api.groq.com/openai/v1"

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model=GROQ_MODEL),
)

In [100]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

-> Response received


-> Response received


/usr/local/python/3.12.1/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [101]:
result.cost

In [102]:
result.all_messages

[EasyInputMessage(content="\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searchers. \n\nAt the end, ask if there are other areas that the user wants to explore.\n", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseReasoningItem(id='resp_01kvp3qj6tft58ed1w62fd3zs1', summary=[], type='reasoning', content=None, encrypted_content=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"Run Olama locally"}', call_id='t4mc2vv3h', name='search', type='function_call', id='t4mc2vv3h', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 't4mc2vv3

In [103]:
result2 = runner.loop(
    prompt='How do I run a different model?',
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


/usr/local/python/3.12.1/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [104]:
runner.run();

-> Response received


-> Response received


-> Response received


/usr/local/python/3.12.1/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


-> Response received
-> Response received
-> Response received
-> Response received
Chat ended.


/usr/local/python/3.12.1/lib/python3.12/site-packages/toyaikit/chat/runners.py:184: UnknownModelWarning: No pricing data for model 'llama-3.1-8b-instant'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(
